# Análise Exploratória de Dados — Pedidos NovaMart

**Nome:** [Seu Nome Completo]  
**Turma:** [Sua Turma]  
**Data:** 26/11/2026  

---

## Apresentação do Problema e Objetivo
A NovaMart disponibilizou uma base fictícia (`pedidos_novamart.csv`) contendo registros de 20.000 pedidos concluídos e entregues. O objetivo desta análise exploratória é compreender a estrutura da base de dados, identificar campos sem preenchimento e investigar valores incomuns (*outliers*) nas variáveis comerciais, logísticas, demográficas e de satisfação dos clientes, apoiando a tomada de decisão da gestão.

## Importação das Bibliotecas e Leitura dos Dados
**Objetivo:** Importar as bibliotecas necessárias para manipulação de dados (`pandas`) e visualização gráfica (`seaborn` e `matplotlib`), configurar o estilo visual dos gráficos e carregar o arquivo CSV.  
**Comandos utilizados:** `pd.read_csv()` realiza a leitura do arquivo separado por vírgulas e ponto decimal, armazenando-o em um DataFrame.

In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid")

df = pd.read_csv("pedidos_novamart.csv")
df.head()

,id_pedido,data_pedido,categoria,regiao,canal_venda,idade_cliente,quantidade_itens,valor_pedido,valor_frete,prazo_entrega_dias,nota_satisfacao,tempo_site_minutos
0,PED-000001,2026-05-31,Eletrônicos,Sul,Aplicativo,18.0,2,315.84,21.44,6,4.0,12.03
1,PED-000002,2026-02-10,Livros,Nordeste,Aplicativo,48.0,4,192.61,32.35,7,2.0,NaN
2,PED-000003,2026-03-14,Esporte,Sul,Aplicativo,39.0,2,511.78,11.86,3,5.0,15.46
3,PED-000004,2026-06-12,Beleza,Nordeste,Aplicativo,27.0,4,253.89,27.42,7,5.0,21.54
4,PED-000005,2026-03-21,Moda,Sul,Aplicativo,53.0,5,243.59,0.00,6,4.0,7.14


## Questão 1 — Estrutura da Base de Dados
**Objetivo:** Verificar as dimensões da tabela (linhas e colunas) e os tipos de dados reconhecidos pelo Pandas para cada variável.  
**Comando utilizado:** `df.info()` exibe um resumo técnico do DataFrame, incluindo índice, nome das colunas, contagem de valores não nulos e tipos (`Dtype`).

In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id_pedido           20000 non-null  object 
 1   data_pedido         20000 non-null  object 
 2   categoria           20000 non-null  object 
 3   regiao              20000 non-null  object 
 4   canal_venda         20000 non-null  object 
 5   idade_cliente       19400 non-null  float64
 6   quantidade_itens    20000 non-null  int64  
 7   valor_pedido        20000 non-null  float64
 8   valor_frete         19600 non-null  float64
 9   prazo_entrega_dias  20000 non-null  int64  
 10  nota_satisfacao     16800 non-null  float64
 11  tempo_site_minutos  19000 non-null  float64
dtypes: float64(5), int64(2), object(5)
memory usage: 1.8+ MB


**Resposta e Insights:**
* **Linhas e colunas:** Existem **20.000 linhas** (índices de `0` a `19999`) e **12 colunas**.
* **Colunas numéricas (7):** `idade_cliente`, `valor_pedido`, `valor_frete`, `nota_satisfacao`, `tempo_site_minutos` (tipo `float64`) e `quantidade_itens`, `prazo_entrega_dias` (tipo `int64`).
* **Colunas textuais (5):** `id_pedido`, `data_pedido`, `categoria`, `regiao` e `canal_venda` (tipo `object`).
* **Tipo da coluna `data_pedido`:** Foi reconhecida como **texto**, pois o tipo apresentado no `info()` é `object` (o Pandas não converte datas automaticamente na leitura padrão sem o parâmetro `parse_dates`).
* **Insight:** Para realizar análises temporais (como sazonalidade por mês ou dia da semana), será necessário converter `data_pedido` para o formato `datetime`.

## Questão 2 — Diagnóstico de Valores Ausentes
**Objetivo:** Identificar quais colunas possuem registros vazios e quantificar exatamente quantos dados faltam em cada variável.  
**Comando utilizado:** `df.info()` permite comparar o `Non-Null Count` de cada coluna com o total de 20.000 linhas (complementado abaixo pela subtração direta com `len(df) - df.count()`).

In [3]:
total_linhas = len(df)
print(f"Total de linhas: {total_linhas}")
print("Registros ausentes por coluna:")
print(total_linhas - df.count())

Total de linhas: 20000
Registros ausentes por coluna:
id_pedido                0
data_pedido              0
categoria                0
regiao                   0
canal_venda              0
idade_cliente          600
quantidade_itens         0
valor_pedido             0
valor_frete            400
prazo_entrega_dias       0
nota_satisfacao       3200
tempo_site_minutos    1000
dtype: int64


**Resposta e Insights:**
* **Colunas com informações ausentes e quantidade de registros vazios:**
  * `nota_satisfacao`: 20.000 − 16.800 = **3.200 registros ausentes** (16,0% da base).
  * `tempo_site_minutos`: 20.000 − 19.000 = **1.000 registros ausentes** (5,0% da base).
  * `idade_cliente`: 20.000 − 19.400 = **600 registros ausentes** (3,0% da base).
  * `valor_frete`: 20.000 − 19.600 = **400 registros ausentes** (2,0% da base).
  * *(As demais 8 colunas possuem todos os 20.000 registros preenchidos).*
* **Diferença entre frete zero e frete sem informação:** Um frete com valor `0.00` representa uma regra comercial válida de **frete grátis**. Já um frete sem informação (`NaN` / nulo) indica **ausência de registro ou falha na captura do dado** no sistema — não permite saber se o pedido teve isenção ou se houve cobrança não gravada.
* **Insight:** Enquanto a ausência de nota de satisfação é esperada (nem todo comprador responde à avaliação), os 400 registros sem `valor_frete` e 1.000 sem `tempo_site_minutos` indicam falhas técnicas de integração que devem ser corrigidas pela TI.

## Questão 3 — Estatísticas Descritivas de `valor_pedido`
**Objetivo:** Avaliar as medidas de tendência central e amplitude dos valores dos pedidos (sem o frete).  
**Comando utilizado:** `df["valor_pedido"].describe()` calcula contagem, média (`mean`), desvio-padrão (`std`), mínimo (`min`), quartis (`25%`, `50%`, `75%`) e máximo (`max`).

In [4]:
df["valor_pedido"].describe()

count    20000.000000
mean       266.550225
std        355.496115
min         19.900000
25%         81.547500
50%        159.140000
75%        314.495000
max       6200.000000
Name: valor_pedido, dtype: float64

**Resposta e Insights:**
* **Média (`mean`):** R$ 266,55
* **Mediana (`50%`):** R$ 159,14
* **Mínimo (`min`):** R$ 19,90
* **Máximo (`max`):** R$ 6.200,00
* **Escolha para comunicar uma compra típica:** A **mediana (R$ 159,14)**. Como existem pedidos muito caros (chegando a R$ 6.200,00), a média é puxada fortemente para cima (R$ 266,55 — 67,5% maior que a mediana). A mediana reflete melhor o comportamento típico do cliente por não ser distorcida por valores extremos.
* **Insight:** Comunicar o ticket médio de R$ 266,55 como padrão de compra geraria expectativas irreais para o time comercial, já que metade dos clientes gasta até R$ 159,14.

## Questão 4 — Quartis e Intervalo Interquartil de `valor_pedido`
**Objetivo:** Compreender como os valores dos pedidos se distribuem em quatro faixas de igual proporção (25% cada) e analisar a dispersão da metade central dos dados.  
**Comando utilizado:** Utiliza-se a saída do `df["valor_pedido"].describe()` obtida anteriormente.

**Resposta e Insights:**
* **1º Quartil (`25%` = R$ 81,55):** 25% dos pedidos tiveram valor de até R$ 81,55 (e 75% custaram acima desse patamar).
* **2º Quartil (`50%` / Mediana = R$ 159,14):** 50% dos pedidos custaram até R$ 159,14 e os outros 50% custaram mais.
* **3º Quartil (`75%` = R$ 314,50):** 75% dos pedidos custaram até R$ 314,50 (apenas os 25% pedidos mais caros superam esse valor).
* **Intervalo entre o 1º e o 3º quartil (IQR):** $314,50 - 81,55 =$ **R$ 232,95** (faixa de **R$ 81,55 a R$ 314,50**). Esse intervalo representa a amplitude de preços onde está concentrada a **metade central (50%)** dos pedidos da loja.
* **Insight:** Campanhas promocionais e faixas de frete grátis estruturadas entre R$ 160,00 e R$ 320,00 atingem diretamente o núcleo central de compras da NovaMart, incentivando o aumento do ticket dos clientes medianos.

## Questão 5 — Histograma de `valor_pedido`
**Objetivo:** Visualizar a frequência dos valores dos pedidos e identificar o formato da distribuição e sua assimetria.  
**Comando utilizado:** `sns.histplot()` com `x="valor_pedido"` e `bins=40` divide o intervalo de preços em 40 faixas para mostrar onde há maior concentração de pedidos.

In [5]:
plt.figure(figsize=(9, 4))
sns.histplot(data=df, x="valor_pedido", bins=40)
plt.title("Distribuição do Valor dos Pedidos")
plt.xlabel("Valor do Pedido (R$)")
plt.ylabel("Quantidade de Pedidos")
plt.show()

**Resposta e Insights:**
* **Faixa de maior concentração:** A maior concentração de pedidos ocorre na faixa inicial do gráfico, aproximadamente **entre R$ 20,00 e R$ 250,00**.
* **Cauda à direita:** Observa-se uma longa cauda estendendo-se para a direita em direção aos valores elevados (acima de R$ 1.000,00 até R$ 6.200,00). Essa assimetria positiva explica por que a média (R$ 266,55) ficou consideravelmente acima da mediana (R$ 159,14).
* **Insight:** A receita da empresa combina alto volume de pedidos de baixo/médio valor com uma cauda longa de pedidos de alto valor que contribuem significativamente para o faturamento total.

## Questão 6 — Boxplot e Valores Atípicos de `valor_pedido`
**Objetivo:** Identificar visualmente a mediana, os quartis e os possíveis valores atípicos (*outliers*) no valor dos pedidos.  
**Comando utilizado:** `sns.boxplot()` com `x="valor_pedido"` desenha a caixa interquartil, os bigodes e os pontos extremos individuais.

In [6]:
plt.figure(figsize=(9, 4))
sns.boxplot(data=df, x="valor_pedido")
plt.title("Boxplot do Valor dos Pedidos")
plt.xlabel("Valor do Pedido (R$)")
plt.show()

**Resposta e Insights:**
* **Leitura do gráfico:** A linha vertical dentro da caixa indica a **mediana (R$ 159,14)**; a caixa delimita os 50% centrais dos pedidos (**R$ 81,55 a R$ 314,50**); o bigode direito estende-se até aproximadamente **R$ 663,92** ($Q3 + 1,5 \times IQR$), e todos os pontos além dele (1.594 pedidos, chegando a **R$ 6.200,00**) são possíveis valores atípicos (*outliers*).
* **Duas hipóteses que merecem investigação:**
  1. Compras legítimas de produtos de alto valor unitário (como *Eletrônicos*) ou compras em grande quantidade no mesmo pedido.
  2. Erro de cadastro de preço no catálogo ou falha de sistema na multiplicação dos itens do pedido.
* **Um valor atípico comprova, sozinho, um erro de cadastro?** **Não.** O *outlier* indica apenas um desvio estatístico em relação ao padrão da maioria; é indispensável cruzar o registro com `categoria` e `quantidade_itens` para confirmar se é uma venda legítima de alto valor ou um erro operacional.

## Questão 7 — Perfil de Idade dos Clientes (`idade_cliente`)
**Objetivo:** Analisar a distribuição etária registrada nos pedidos e compreender as limitações dessa variável.  
**Comandos utilizados:** `df["idade_cliente"].describe()` para métricas descritivas e `sns.histplot()` com `bins=20` para observar as faixas etárias mais frequentes.

In [7]:
print(df["idade_cliente"].describe())

plt.figure(figsize=(9, 4))
sns.histplot(data=df, x="idade_cliente", bins=20)
plt.title("Distribuição da Idade dos Clientes por Pedido")
plt.xlabel("Idade do Cliente (anos)")
plt.ylabel("Quantidade de Pedidos")
plt.show()

count    19400.000000
mean        38.568402
std         13.366121
min         18.000000
25%         28.000000
50%         36.000000
75%         48.000000
max         80.000000
Name: idade_cliente, dtype: float64


**Resposta e Insights:**
* **Estatísticas:** Idade mínima de **18 anos**, máxima de **80 anos**, média de **38,57 anos** e mediana de **36,00 anos**.
* **Faixas mais frequentes:** A maior concentração de compras ocorre entre **25 e 45 anos** (com pico na faixa de 26 a 35 anos, seguida de 36 a 45 anos), reduzindo-se gradualmente acima dos 50 anos. Não há registros de menores de 18 anos ou idades impossíveis.
* **Limitações:** Existem **600 pedidos sem idade informada** e um mesmo cliente pode aparecer diversas vezes na base. Portanto, o gráfico representa o perfil etário **dos 19.400 pedidos com idade preenchida**, e não uma contagem de clientes únicos cadastrados.
* **Insight:** O público jovem-adulto (25 a 45 anos) concentra a maior demanda da loja, devendo ser o foco principal das campanhas de aquisição e retenção.

## Questão 8 — Análise do Prazo de Entrega (`prazo_entrega_dias`)
**Objetivo:** Avaliar o tempo efetivo de entrega dos pedidos, sua concentração e a presença de atrasos extremos.  
**Comandos utilizados:** `df["prazo_entrega_dias"].describe()`, `sns.histplot()` (`bins=20`) e `sns.boxplot()` exibidos lado a lado.

In [8]:
print(df["prazo_entrega_dias"].describe())

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(data=df, x="prazo_entrega_dias", bins=20, ax=axes[0])
axes[0].set_title("Histograma — Prazo de Entrega")
axes[0].set_xlabel("Prazo de Entrega (dias corridos)")
axes[0].set_ylabel("Quantidade de Pedidos")

sns.boxplot(data=df, x="prazo_entrega_dias", ax=axes[1])
axes[1].set_title("Boxplot — Prazo de Entrega")
axes[1].set_xlabel("Prazo de Entrega (dias corridos)")
plt.tight_layout()
plt.show()

count    20000.00000
mean         5.78410
std          4.40472
min          1.00000
25%          4.00000
50%          5.00000
75%          7.00000
max         40.00000
Name: prazo_entrega_dias, dtype: float64


**Resposta e Insights:**
* **Estatísticas:** Média de **5,78 dias**, mediana de **5,00 dias** e máximo de **40 dias** (com 50% centrais entre 4 e 7 dias).
* **Concentração e valores atípicos:** A grande maioria das entregas concentra-se entre **3 e 8 dias corridos**, mas existe uma cauda longa à direita com **497 outliers acima de 11 dias**, atingindo até **40 dias**.
* **Podemos afirmar que houve descumprimento do prazo prometido apenas com essa coluna?** **Não.** Segundo o dicionário de dados, `prazo_entrega_dias` registra apenas o *tempo efetivamente decorrido* entre a compra e a entrega, não havendo informação sobre qual foi o *prazo prometido* ao consumidor no ato da compra.
* **Insight:** Embora a operação logística padrão seja ágil (mediana de 5 dias), quase 500 pedidos levaram de 12 a 40 dias, sinalizando ocorrências críticas que exigem auditoria por transportadora e região.

## Questão 9 — Análise do Valor do Frete (`valor_frete`)
**Objetivo:** Examinar a distribuição dos valores cobrados de frete, o impacto do frete grátis e os valores atípicos.  
**Comandos utilizados:** `df["valor_frete"].describe()` e `sns.boxplot()` com `x="valor_frete"`.

In [9]:
print(df["valor_frete"].describe())

plt.figure(figsize=(9, 4))
sns.boxplot(data=df, x="valor_frete")
plt.title("Distribuição do Valor do Frete")
plt.xlabel("Valor do Frete (R$)")
plt.show()

count    19600.000000
mean        19.957685
std         13.385942
min          0.000000
25%         14.450000
50%         21.070000
75%         26.820000
max        118.560000
Name: valor_frete, dtype: float64


**Resposta e Insights:**
* **Estatísticas:** Mínimo de **R$ 0,00**, mediana de **R$ 21,07**, 3º quartil de **R$ 26,82** e máximo de **R$ 118,56** (1º quartil em R$ 14,45 e média em R$ 19,96).
* **Dispersão e valores atípicos:** Metade central dos pedidos tem frete entre **R$ 14,45 e R$ 26,82**. Existem **195 valores atípicos** acima de R$ 45,38, chegando a **R$ 118,56** (possivelmente mercadorias pesadas/volumosas ou destinos de difícil acesso).
* **Interpretação do frete grátis e das informações ausentes:** Há **3.325 pedidos** com frete igual a `R$ 0,00` (benefício real de frete grátis, que puxa o mínimo para zero e a média para baixo da mediana). Já os **400 registros ausentes** são desconsiderados nos cálculos e no gráfico, representando falta de informação no sistema (não devem ser interpretados como frete grátis).
* **Insight:** O frete grátis está presente em cerca de 16,6% da base e ajuda a conter o custo médio de envio, mas fretes acima de R$ 45,00 podem gerar abandono de carrinho se não houver subsídio.

## Questão 10 — Avaliação de Satisfação (`nota_satisfacao`)
**Objetivo:** Analisar a distribuição das notas atribuídas pelos clientes e avaliar a representatividade das respostas.  
**Comandos utilizados:** `df["nota_satisfacao"].describe()` e `sns.histplot()` com `discrete=True` para alinhar cada barra exatamente aos valores inteiros de 1 a 5.

In [10]:
print(df["nota_satisfacao"].describe())

plt.figure(figsize=(8, 4))
sns.histplot(data=df, x="nota_satisfacao", discrete=True)
plt.title("Distribuição das Notas de Satisfação")
plt.xlabel("Nota de Satisfação (1 = Muito Insatisfeito a 5 = Muito Satisfeito)")
plt.ylabel("Quantidade de Avaliações")
plt.xticks([1, 2, 3, 4, 5])
plt.show()

count    16800.000000
mean         4.051190
std          0.817904
min          1.000000
25%          4.000000
50%          4.000000
75%          5.000000
max          5.000000
Name: nota_satisfacao, dtype: float64


**Resposta e Insights:**
* **Estatísticas:** Média de **4,05**, mediana de **4,00** e total de **16.800 avaliações** registradas.
* **Notas visualmente mais frequentes:** As notas **4** (7.681 avaliações) e **5** (5.321 avaliações) são amplamente predominantes, seguidas pela nota **3** (3.222). Notas baixas são minoria: nota **2** (489) e nota **1** (87).
* **Comparação do `count` e representatividade:** Dos 20.000 pedidos, **16.800 possuem avaliação** e **3.200 (16,0%) estão sem nota**. Não podemos afirmar que essas avaliações representam a opinião de todos os compradores devido ao **viés de não resposta (autosseleção)**: clientes que não avaliam podem ter tido experiências neutras ou insatisfatórias sem motivação para responder, tornando arriscado extrapolar a média 4,05 para toda a base.
* **Insight:** Embora 77,4% dos respondentes deem notas 4 ou 5, investigar os 576 pedidos com notas 1 ou 2 (cruzando com atrasos de entrega) pode revelar rapidamente as principais causas de insatisfação.

## Questão 11 — Tamanho dos Pedidos (`quantidade_itens`)
**Objetivo:** Investigar a quantidade de unidades adquiridas por pedido e identificar padrões de consumo ou compras atípicas.  
**Comandos utilizados:** `df["quantidade_itens"].describe()` e `sns.boxplot()` com `x="quantidade_itens"` para evidenciar a concentração e os outliers.

In [11]:
print(df["quantidade_itens"].describe())

plt.figure(figsize=(9, 4))
sns.boxplot(data=df, x="quantidade_itens")
plt.title("Distribuição da Quantidade de Itens por Pedido")
plt.xlabel("Quantidade de Itens")
plt.show()

count    20000.000000
mean         2.642300
std          2.390008
min          1.000000
25%          1.000000
50%          2.000000
75%          4.000000
max         20.000000
Name: quantidade_itens, dtype: float64


**Resposta e Insights:**
* **Estatísticas:** Mediana de **2,00 itens**, 3º quartil de **4,00 itens** e máximo de **20 itens** (1º quartil e mínimo em 1 item; média em 2,64 itens).
* **Tamanho dos pedidos e situações para investigação:** Como 75% dos pedidos têm entre **1 e 4 itens**, o perfil predominante da NovaMart é de **varejo individual** (compras pequenas). Contudo, existem **388 pedidos atípicos** acima de 8 unidades (chegando a **20 itens**), que merecem investigação para diferenciar compras de revendedores/corporativas (B2B) de eventuais erros de digitação na quantidade do carrinho.
* **Insight:** Criar kits ou descontos progressivos a partir de 3 ou 4 unidades pode elevar o tamanho médio da cesta sem exigir grandes investimentos em aquisição de novos clientes.

## Questão 12 — Conclusão Executiva para o Gestor (Ricardo)

Prezado **Ricardo**,

A análise exploratória dos **20.000 pedidos** registrados na base `pedidos_novamart.csv` fornece um panorama claro do desempenho comercial, logístico e de qualidade de dados da operação.

### 1. Principais Evidências Observadas nos Dados
* **Perfil de Compra (Varejo Típico vs. Cauda Longa):** Metade dos pedidos custa até **R$ 159,14** (mediana) e contém até **2 itens** (75% têm até 4 itens). No entanto, **1.594 pedidos atípicos** superam R$ 663,92 — atingindo até **R$ 6.200,00** —, o que puxa o ticket médio para **R$ 266,55**.
* **Desempenho Logístico e Frete:** A entrega típica leva **5 dias corridos** (mediana), com 75% concluídas em até **7 dias**, porém **497 pedidos** ultrapassaram 11 dias (chegando a **40 dias**). O frete mediano é de **R$ 21,07**, com **3.325 pedidos** (16,6%) utilizando frete grátis (`R$ 0,00`) e picos de até **R$ 118,56**.
* **Público e Satisfação:** Nos pedidos com idade informada, há forte concentração entre **25 e 45 anos** (mediana de **36 anos**). Entre os 16.800 pedidos avaliados, a satisfação é positiva (média **4,05** e mediana **4,00**), liderada pelas notas 4 e 5.
* **Integridade dos Registros:** Foram identificados campos sem preenchimento em quatro colunas: `nota_satisfacao` (**3.200 vazios / 16,0%**), `tempo_site_minutos` (**1.000 / 5,0%**), `idade_cliente` (**600 / 3,0%**) e `valor_frete` (**400 / 2,0%**).

### 2. Recomendações Práticas
1. **Corrigir falhas de captura no checkout:** Auditar junto à equipe de tecnologia o motivo de **400 pedidos** não registrarem `valor_frete` e **1.000** não registrarem `tempo_site_minutos`, garantindo integridade nas métricas financeiras e de navegação.
2. **Investigar gargalos logísticos e pedidos de alto volume:** Mapear os **497 pedidos** com entrega superior a 11 dias por região e transportadora para atuar nos atrasos extremos, e auditar os pedidos acima de R$ 1.000,00 ou 10+ itens para identificar clientes corporativos/revendedores ou possíveis falhas de precificação.
3. **Aprimorar o monitoramento de SLA e avaliações:** Passar a gravar o *prazo prometido* no momento da compra (permitindo medir atrasos reais) e implementar disparos automáticos de pesquisa pós-entrega para reduzir os 16% de pedidos sem avaliação.

### 3. Limitações da Análise (Fatos Observados vs. Hipóteses)
* **Contagem de pedidos vs. clientes únicos:** Como a base não possui identificador único de cliente, um mesmo comprador pode ter feito vários pedidos; assim, a distribuição etária reflete o volume de pedidos por faixa etária, e não o número de pessoas distintas.
* **Atrasos efetivos vs. descumprimento de prazo:** Entregas de até 40 dias são um **fato observado** nos dados, mas afirmar que houve *descumprimento do prazo prometido* é apenas uma **hipótese**, pois a base não informa qual prazo foi acordado com o cliente.
* **Origem dos *outliers*:** Os valores extremos de R$ 6.200,00 ou 20 itens são **evidências estatísticas**, mas associá-los a compras B2B ou erros de sistema permanece como **hipótese** até a auditoria individual dos itens de cada pedido.
* **Viés de avaliação:** A média de satisfação (4,05) considera apenas os 84% de pedidos avaliados; não se deve assumir automaticamente que os 16% sem nota compartilham do mesmo nível de satisfação.